# 🍷 Wine Quality Classifier

## What This Project Does
In this project we will build a machine learning model that classifies a bottle of red wine as either **good** (quality ≥ 7) or **bad** (quality < 7) based on its **chemical properties** (acidity, sugar, alcohol content, etc.).

We will train and compare **three different models** — K-Nearest Neighbors (KNN), Support Vector Machine (SVM), and Random Forest — to see which one performs best. We will also use **hyperparameter tuning** to squeeze extra performance out of the best model.

## What You Will Learn
- How to **bin a continuous target** into a binary label (good vs bad)
- Why some models (KNN, SVM) need **feature scaling** while others (Random Forest) do not
- How **three different classifiers** work at a high level and when to use each
- How to use **GridSearchCV** for hyperparameter tuning
- How to read and interpret a **feature importance plot**

## Dataset
- **Source:** [Red Wine Quality on Kaggle](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)
- **File to upload:** `winequality-red.csv`

> 💡 **Tip:** Download the dataset from Kaggle before running this notebook.

In [ ]:
# ── STEP 1: Install Libraries ─────────────────────────────────────────────────
# All required libraries (pandas, numpy, scikit-learn, matplotlib, seaborn)
# come pre-installed in Google Colab, so there is nothing extra to install.
# We still run this cell to confirm everything is ready.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

sns.set_theme(style='whitegrid')
print('✅ All libraries imported!')

In [ ]:
# ── STEP 2: Upload the Dataset ────────────────────────────────────────────────
# Click 'Choose Files' and upload winequality-red.csv.

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Uploaded: {filename}')

In [ ]:
# ── STEP 2 (continued): Load the Dataset ──────────────────────────────────────
# The CSV uses semicolons (;) as separators instead of the usual commas.
# We tell pandas about this with the 'sep' argument.

df = pd.read_csv(filename, sep=';')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── STEP 3: Create a Binary Target Label ─────────────────────────────────────
# The original 'quality' column is a score from 3 to 8.
# Classifying 6 separate classes would be harder and less useful.
# Instead, we make a simpler 'good wine vs bad wine' binary problem:
#   • quality >= 7 → 'good wine' → label 1
#   • quality  < 7 → 'bad wine'  → label 0
#
# Why 7? It's the commonly used threshold in the wine quality literature —
# wines rated 7 or above are considered 'high quality' by industry standards.
# Turning a continuous score into a binary label is called 'binarization'.

df['good_wine'] = (df['quality'] >= 7).astype(int)   # True → 1, False → 0
df = df.drop(columns=['quality'])                    # Remove the original quality column

print('Target distribution:')
print(df['good_wine'].value_counts())
print(f'\n{df["good_wine"].value_counts()[1]} good wines vs {df["good_wine"].value_counts()[0]} bad wines')

In [ ]:
# ── STEP 4: Exploratory Data Analysis (EDA) ───────────────────────────────────
# We visualize the data before modelling to spot patterns and potential issues.

# Plot 4a: Distribution of good vs bad wine
plt.figure(figsize=(5, 4))
target_counts = df['good_wine'].value_counts()
sns.barplot(x=['Bad Wine (0)', 'Good Wine (1)'], y=target_counts.values, palette='muted')
plt.title('Class Balance: Good vs Bad Wine')
plt.ylabel('Count')
for i, v in enumerate(target_counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 4 (continued): Correlation Heatmap ───────────────────────────────────
# A correlation heatmap shows how strongly every pair of features is linearly
# related. Values close to +1 or -1 indicate strong correlation; values near 0
# indicate no linear relationship. If two input features are highly correlated
# with each other (multicollinearity), they are providing redundant information.

plt.figure(figsize=(11, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # Only show lower triangle
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 5: Feature and Target Split ─────────────────────────────────────────

X = df.drop(columns=['good_wine'])   # Input features (chemical properties)
y = df['good_wine']                  # Target: 1 = good wine, 0 = bad wine

feature_names = list(X.columns)
print(f'Features: {feature_names}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# ── STEP 6: Feature Scaling ───────────────────────────────────────────────────
# Different models have different relationships with feature scale:
#
# ✅ KNN (K-Nearest Neighbors) NEEDS scaling:
#    KNN measures the distance between data points. If one feature has values
#    in the hundreds (e.g. 'total sulfur dioxide') and another has values around
#    0-1 (e.g. 'volatile acidity'), the large-scale feature will dominate the
#    distance calculation and make smaller features invisible to the algorithm.
#
# ✅ SVM (Support Vector Machine) NEEDS scaling:
#    SVM also relies on distances (it finds the maximum-margin hyperplane).
#    Unscaled features cause the margin to be skewed toward high-magnitude features.
#
# ❌ Random Forest does NOT need scaling:
#    Decision trees split features based on threshold values (e.g. 'if alcohol > 11').
#    This doesn't depend on the scale of the feature, only its order/rank.
#    Scaling wouldn't change the tree's splitting decisions.
#
# We scale anyway so the same X_scaled is valid for all three models.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaling applied.')

In [ ]:
# ── STEP 7: Train / Test Split ────────────────────────────────────────────────
# 80% training, 20% testing. stratify=y preserves the class ratio.

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
# ── STEP 8a: Model 1 — K-Nearest Neighbors (KNN) ─────────────────────────────
# KNN is one of the simplest ML algorithms. To classify a new data point, it:
#   1. Calculates the distance to every training point
#   2. Finds the K nearest neighbors (K=5 by default)
#   3. Takes a majority vote among those K neighbors and returns that class
#
# Pros: simple, no 'training' step, works well on small datasets
# Cons: slow at prediction time on large datasets; sensitive to irrelevant features

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

knn_acc = accuracy_score(y_test, knn.predict(X_test))
print(f'KNN Accuracy: {knn_acc:.4f} ({knn_acc*100:.1f}%)')

In [ ]:
# ── STEP 8b: Model 2 — Support Vector Machine (SVM) ──────────────────────────
# SVM finds the hyperplane (decision boundary) that maximizes the margin
# between the two classes. It focuses on the 'support vectors' — the training
# points closest to the boundary — and ignores points far away.
#
# With kernel='rbf' (Radial Basis Function), SVM can find non-linear boundaries
# by mapping the data to a higher-dimensional space.
#
# Pros: very effective in high-dimensional spaces; memory-efficient
# Cons: slow on large datasets; hard to interpret; requires scaling

svm = SVC(kernel='rbf', random_state=42, probability=True)
svm.fit(X_train, y_train)

svm_acc = accuracy_score(y_test, svm.predict(X_test))
print(f'SVM Accuracy: {svm_acc:.4f} ({svm_acc*100:.1f}%)')

In [ ]:
# ── STEP 8c: Model 3 — Random Forest ─────────────────────────────────────────
# Random Forest builds many decision trees, each on a random subset of data and
# features, then combines their votes (bagging). This reduces overfitting and
# gives feature importance scores 'for free'.
#
# Pros: handles non-linear relationships; robust to outliers; gives feature importances
# Cons: less interpretable than a single tree; slower than KNN on small data

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.1f}%)')

In [ ]:
# ── STEP 9: Compare All Three Models ─────────────────────────────────────────
# We collect accuracy, precision, recall, and F1 for each model in a DataFrame
# so we can compare them side by side.
#
# Precision: Of all wines predicted as 'good', what % are actually good?
# Recall:    Of all wines that are actually good, what % did the model catch?
# F1-score:  Harmonic mean of precision and recall — useful when classes are unbalanced

models = {'KNN': knn, 'SVM': svm, 'Random Forest': rf}
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1'       : round(f1_score(y_test, y_pred), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print('=== Model Comparison Table ===')
print(results_df.to_string())

In [ ]:
# ── STEP 10: Hyperparameter Tuning with GridSearchCV ─────────────────────────
# 'Hyperparameters' are the settings you configure BEFORE training a model —
# things like how many trees to build, or how deep each tree can grow.
# These are different from 'parameters' (weights) which the model learns automatically.
#
# GridSearchCV works by:
#   1. You give it a grid of hyperparameter values to try
#   2. It trains the model on every combination in the grid
#   3. It evaluates each using cross-validation (splits training data into k folds,
#      trains on k-1 folds, tests on 1, and repeats k times)
#   4. It returns the combination that gave the best cross-validation score
#
# We tune Random Forest since it tends to benefit most from tuning.
# cv=3 means 3-fold cross-validation (keeps it fast). n_jobs=-1 uses all CPU cores.

param_grid = {
    'n_estimators': [50, 100, 200],    # Number of trees to build
    'max_depth'   : [None, 5, 10],     # Maximum depth of each tree (None = unlimited)
    'min_samples_split': [2, 5],       # Minimum samples needed to split a node
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',    # Optimize for F1 score (good for unbalanced classes)
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

best_rf = grid_search.best_estimator_
tuned_acc = accuracy_score(y_test, best_rf.predict(X_test))
print(f'Tuned Random Forest Test Accuracy: {tuned_acc:.4f}')

In [ ]:
# ── STEP 11: Feature Importance Plot ─────────────────────────────────────────
# Random Forest gives us a feature importance score for each input feature.
# This score reflects how much each feature reduced impurity (uncertainty)
# across all trees on average. Features with higher importance had a greater
# impact on reducing prediction errors.
#
# This tells winemakers and researchers which chemical properties are most
# predictive of wine quality — very useful domain knowledge.

importances = best_rf.feature_importances_
importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=True)
)

plt.figure(figsize=(8, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Feature Importance (Mean Decrease in Impurity)')
plt.title('Random Forest Feature Importance — Wine Quality')
plt.tight_layout()
plt.show()
print(f'Top feature: {importance_df.iloc[-1]["Feature"]}')

## 🎉 Summary: What You Learned

Congratulations on completing the **Wine Quality Classifier** project! Here's a recap:

| Step | What You Did | Why It Matters |
|------|--------------|----------------|
| Binarization | Converted quality score to good/bad label | Simplifies a hard multi-class problem into binary |
| Correlation Heatmap | Plotted feature correlations | Reveals redundant or strongly related features |
| Scaling | Applied StandardScaler | Required by KNN and SVM; safe to apply to Random Forest too |
| 3 Models | Trained KNN, SVM, Random Forest | Compare different algorithmic approaches |
| Model Comparison | Accuracy, Precision, Recall, F1 table | Shows strengths and weaknesses of each model |
| GridSearchCV | Tuned Random Forest hyperparameters | Squeezes extra performance out of the best model |
| Feature Importance | Plotted which chemicals matter most | Connects ML results back to real-world domain knowledge |

### Next Steps
- Try **multi-class classification** by keeping all quality scores (3–8) instead of binarizing
- Try **XGBoost or LightGBM** for potentially better accuracy
- Extend the analysis to the **white wine** dataset and compare results